# ASO → IDT order format

Turn an antisense oligonucleotide — **sequence** + per-position **sugar chemistry** + **phosphorothioate (PS) pattern** — into an IDT-style order string, using the shared `tauso.common.modifications.to_idt_notation`.

- **Chemical (sugar) pattern** — one letter per base, so its length equals the sequence length: `M` = 2′-MOE · `C` = cEt · `L` = LNA · `d` = DNA
- **PS pattern** — one character per inter-nucleotide linkage, so its length equals *sequence length − 1*: `*` = phosphorothioate · anything else = phosphodiester. Omit it for an all-PS backbone.

`to_idt_notation` validates both lengths (chemical pattern = *L*, PS pattern = *L − 1*) and raises `ValueError` on a mismatch or an unsupported base/sugar.

> ⚠️ The emitted modification codes are a starting point — confirm them against IDT's current catalogue before ordering (cEt in particular is a placeholder token).

In [1]:
from tauso.common.modifications import to_idt_notation

def standard_gapmer(sequence, chemistry="MOE"):
    """Shortcut chemical pattern for a standard gapmer: modified wings + DNA gap, with the
    reasonable default wing size per chemistry (2'-MOE -> 5-nt wings, e.g. a 20-mer is 5-10-5;
    cEt -> 3-nt wings, e.g. a 16-mer is 3-10-3). `chemistry` defaults to 2'-MOE."""
    wing, code = {"MOE": (5, "M"), "cEt": (3, "C")}[chemistry]
    return code * wing + "d" * (len(sequence) - 2 * wing) + code * wing

## One ASO

Type a **sequence** and a **chemical pattern** (one sugar letter per base), then run. For a standard 2′-MOE or cEt gapmer you can skip typing the pattern and use `standard_gapmer(sequence)` / `standard_gapmer(sequence, "cEt")` instead.

In [2]:
sequence         = "GCAGTTATATTAGGTTCTCG"       # bases, 5'->3'
chemical_pattern = "MMMMMddddddddddMMMMM"        # one sugar per base: M=2'MOE  C=cEt  L=LNA  d=DNA
ps_pattern       = "*" * (len(sequence) - 1)     # optional; all-PS by default

print(to_idt_notation(sequence, chemical_pattern, ps_pattern))

# shortcut — a standard 2'-MOE gapmer gives the same pattern without typing it out:
print(to_idt_notation(sequence, standard_gapmer(sequence)))

/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/
/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOErT/*T*A*T*A*T*T*A*G*G*T*/i2MOErT/*/i2MOErC/*/i2MOErT/*/i2MOErC/*/32MOErG/


## A batch of ASOs

One row per ASO — type the chemical pattern explicitly, or use `standard_gapmer(...)`. `to_idt_notation` defaults to an all-PS backbone.

In [3]:
import pandas as pd

# (name, sequence, chemical_pattern)
specs = [
    ("MALAT1_top", "GCAGTTATATTAGGTTCTCG", standard_gapmer("GCAGTTATATTAGGTTCTCG")),         # 2'-MOE 5-10-5
    ("v2_3",       "GTTATATTAGGTTCTCGTGT", "MMMMMddddddddddMMMMM"),                          # explicit
    ("cet_16",     "GCATTGGTATTCATGA",     standard_gapmer("GCATTGGTATTCATGA", "cEt")),      # cEt 3-10-3
]

pd.DataFrame(
    [{"name": n, "sequence": s, "chemical_pattern": p, "idt_order": to_idt_notation(s, p)}
     for n, s, p in specs]
)

,name,sequence,chemical_pattern,idt_order
0,MALAT1_top,GCAGTTATATTAGGTTCTCG,MMMMMddddddddddMMMMM,/52MOErG/*/i2MOErC/*/i2MOErA/*/i2MOErG/*/i2MOE...
1,v2_3,GTTATATTAGGTTCTCGTGT,MMMMMddddddddddMMMMM,/52MOErG/*/i2MOErT/*/i2MOErT/*/i2MOErA/*/i2MOE...
2,cet_16,GCATTGGTATTCATGA,CCCddddddddddCCC,/icEtG/*/icEtC/*/icEtA/*T*T*G*G*T*A*T*T*C*A*/i...
